In [ ]:
from litellm import completion
from dotenv import load_dotenv
import os

import pdfplumber
import json

load_dotenv()
MODEL = "ollama/qwen2.5:7b-instruct"
CAPITULOS_PROMPT = """
Leia o pdf e separe o texto em capítulos.
Seu output deve ser SOMENTE um JSON exatamente neste formato:
{
    "1 Informações Gerais": "",
    "2 Apresentação": "",
    "3 Exposição de Motivos": "",
    "4 Objetivos": "",
    "5 Princípios Norteadores para a Formação Profissional": "",
    "6 Expectativas da Formação Profissional": "",
    "7 Estrutura Curricular": "",
    "8 Estágio Curricular": "",
    "9 Trabalho de Conclusão de Curso": "",
    "10 Atividades Complementares": "",
    "11 Integração Ensino, Pesquisa e Extensão": "",
    "12 Avaliação do Processo de Ensino-Aprendizagem": "",
    "13 Avaliação do Projeto de Curso": "",
    "14 Qualificação de Docentes e Técnico-Administrativos": "",
    "15 Requisitos Legais e Normativos": "",
    "16 Dinâmica das Atividades (EAD)": "",
    "17 Referências": "",
    "18 Apêndices": "",
    "19 Extras": ""
}
Caso algum capítulo não exista, escreva "não existe".
Se existir algum capítulo diferente da lista, coloque em "19 Extras".
Retorne SOMENTE o JSON.
"""

def generate_response(messages, tools=None):
    response = completion(
        model=MODEL,
        messages=messages,
        tools=tools,
        api_base = os.getenv('OLLAMA_API_BASE'),
    )
    return response

def verificar_response(ai_response, regra):
    system_prompt = [{
        "role": "system",
        "content":
        "Você é um agente verificador de respostas.\n"
        f"Verifique se o output segue exatamente estas regras:\n\n{regra}\n\n"
        'Responda SOMENTE "certo" ou "errado".'
    }]
    user_prompt = [{
        "role": "user",
        "content": ai_response
    }]
    while True:

        response = generate_response(system_prompt + user_prompt)
        status = (
            response
            .choices[0]
            .message
            .content
            .strip()
            .lower()
        )
        if status == "certo":
            return True
        if status == "errado":
            return False

def ler_texto_pdf(arquivo):
    texto = ""
    with pdfplumber.open(arquivo) as pdf:
        for pagina in pdf.pages:
            texto += (pagina.extract_text() or "") + "\n"
    return texto


def limpar_json(texto):
    texto = texto.strip()
    if texto.startswith("```json"):
        texto = texto.replace("```json", "", 1)
    if texto.endswith("```"):
        texto = texto[:-3]
    return texto.strip()


def separar_capitulos(pdf):
    texto = ler_texto_pdf(pdf)
    messages = [
        {
            "role": "system",
            "content": CAPITULOS_PROMPT
        },
        {
            "role": "user",
            "content": texto
        }
    ]
    for tentativa in range(3):
        try:

            response = generate_response(messages)
            resultado = response.choices[0].message.content
            resultado = limpar_json(resultado)
            if verificar_response(resultado, CAPITULOS_PROMPT):
                return json.loads(resultado)
        except Exception as e:
            print(f"Tentativa {tentativa+1}: {e}")

    return None


print(separar_capitulos('ppcesmola.pdf'))